# OpenWeatherMap NFL Weather Ingestion

Pull current and forecast weather data for NFL stadiums from OpenWeatherMap API.

**Features:**
- Free tier: 1,000 API calls/day, 60 calls/minute
- Current weather + 5-day/3-hour forecast
- Wind speed, gusts, direction
- Temperature and precipitation
- Global coverage

**Weather Metrics for Fantasy:**
- Wind speed/gusts → QB, K, deep WR downgrade
- Temperature → Cold weather RB boost
- Precipitation → Run-heavy game scripts
- Wind direction → Crosswind kicker impact

**Resources:**
- Website: https://openweathermap.org
- API Docs: https://openweathermap.org/api
- Free tier: 1K calls/day (60/min)
- Sign up: https://home.openweathermap.org/users/sign_up

In [0]:
import requests
import pandas as pd
import json
from datetime import datetime, timedelta
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Configuration
BASE_URL = "https://api.openweathermap.org/data/2.5"
SEASON = 2024
WEEK = 18

# Get API key from Databricks Secrets
try:
    API_KEY = dbutils.secrets.get(scope="api-keys", key="openweathermap-key")
    print("✓ API key loaded from secrets")
except:
    print("⚠️ ERROR: API key not found in secrets")
    print("\nTo set up:")
    print("1. Sign up at https://home.openweathermap.org/users/sign_up")
    print("2. Get your API key (activation takes ~10min-2hrs)")
    print("3. Add to secrets: databricks secrets put --scope api-keys --key openweathermap-key")
    print("4. Or set manually: API_KEY = 'your-key-here'")
    # Uncomment to set manually
    # API_KEY = "your-openweathermap-key-here"
    raise

print(f"\n☀️ OpenWeatherMap NFL Weather Ingestion")
print(f"Season: {SEASON}, Week: {WEEK}")
print(f"API Endpoint: {BASE_URL}")
print(f"Free Tier: 1,000 calls/day (60/min)")

In [0]:
# NFL stadium lat/long coordinates for OpenWeatherMap
# OpenWeatherMap works best with coordinates for accuracy

NFL_STADIUM_COORDS = {
    'ARI': {'city': 'Glendale, AZ', 'lat': 33.5276, 'lon': -112.2626, 'is_dome': True},
    'ATL': {'city': 'Atlanta, GA', 'lat': 33.7554, 'lon': -84.4008, 'is_dome': True},
    'BAL': {'city': 'Baltimore, MD', 'lat': 39.2780, 'lon': -76.6227, 'is_dome': False},
    'BUF': {'city': 'Orchard Park, NY', 'lat': 42.7738, 'lon': -78.7870, 'is_dome': False},
    'CAR': {'city': 'Charlotte, NC', 'lat': 35.2258, 'lon': -80.8529, 'is_dome': False},
    'CHI': {'city': 'Chicago, IL', 'lat': 41.8623, 'lon': -87.6167, 'is_dome': False},
    'CIN': {'city': 'Cincinnati, OH', 'lat': 39.0954, 'lon': -84.5160, 'is_dome': False},
    'CLE': {'city': 'Cleveland, OH', 'lat': 41.5061, 'lon': -81.6995, 'is_dome': False},
    'DAL': {'city': 'Arlington, TX', 'lat': 32.7473, 'lon': -97.0945, 'is_dome': True},
    'DEN': {'city': 'Denver, CO', 'lat': 39.7439, 'lon': -105.0201, 'is_dome': False},
    'DET': {'city': 'Detroit, MI', 'lat': 42.3400, 'lon': -83.0456, 'is_dome': True},
    'GB': {'city': 'Green Bay, WI', 'lat': 44.5013, 'lon': -88.0622, 'is_dome': False},
    'HOU': {'city': 'Houston, TX', 'lat': 29.6847, 'lon': -95.4107, 'is_dome': True},
    'IND': {'city': 'Indianapolis, IN', 'lat': 39.7601, 'lon': -86.1639, 'is_dome': True},
    'JAX': {'city': 'Jacksonville, FL', 'lat': 30.3240, 'lon': -81.6373, 'is_dome': False},
    'KC': {'city': 'Kansas City, MO', 'lat': 39.0489, 'lon': -94.4839, 'is_dome': False},
    'LAC': {'city': 'Inglewood, CA', 'lat': 33.9535, 'lon': -118.3390, 'is_dome': False},
    'LAR': {'city': 'Inglewood, CA', 'lat': 33.9535, 'lon': -118.3390, 'is_dome': False},
    'LV': {'city': 'Las Vegas, NV', 'lat': 36.0909, 'lon': -115.1833, 'is_dome': True},
    'MIA': {'city': 'Miami Gardens, FL', 'lat': 25.9580, 'lon': -80.2389, 'is_dome': False},
    'MIN': {'city': 'Minneapolis, MN', 'lat': 44.9738, 'lon': -93.2577, 'is_dome': True},
    'NE': {'city': 'Foxborough, MA', 'lat': 42.0909, 'lon': -71.2643, 'is_dome': False},
    'NO': {'city': 'New Orleans, LA', 'lat': 29.9511, 'lon': -90.0812, 'is_dome': True},
    'NYG': {'city': 'East Rutherford, NJ', 'lat': 40.8128, 'lon': -74.0742, 'is_dome': False},
    'NYJ': {'city': 'East Rutherford, NJ', 'lat': 40.8128, 'lon': -74.0742, 'is_dome': False},
    'PHI': {'city': 'Philadelphia, PA', 'lat': 39.9008, 'lon': -75.1675, 'is_dome': False},
    'PIT': {'city': 'Pittsburgh, PA', 'lat': 40.4468, 'lon': -80.0158, 'is_dome': False},
    'SEA': {'city': 'Seattle, WA', 'lat': 47.5952, 'lon': -122.3316, 'is_dome': False},
    'SF': {'city': 'Santa Clara, CA', 'lat': 37.4032, 'lon': -121.9698, 'is_dome': False},
    'TB': {'city': 'Tampa, FL', 'lat': 27.9759, 'lon': -82.5033, 'is_dome': False},
    'TEN': {'city': 'Nashville, TN', 'lat': 36.1665, 'lon': -86.7713, 'is_dome': False},
    'WAS': {'city': 'Landover, MD', 'lat': 38.9078, 'lon': -76.8645, 'is_dome': False}
}

print(f"Loaded {len(NFL_STADIUM_COORDS)} NFL stadium coordinates")
outdoor_stadiums = sum(1 for s in NFL_STADIUM_COORDS.values() if not s['is_dome'])
print(f"Outdoor stadiums: {outdoor_stadiums}")
print(f"Domed stadiums: {len(NFL_STADIUM_COORDS) - outdoor_stadiums}")

In [0]:
# Fetch current weather for all NFL stadiums
import time

print("Fetching current weather for NFL stadiums...\n")

current_weather_data = []
rate_limit_delay = 1.1  # Stay under 60 calls/min

for team, stadium_info in NFL_STADIUM_COORDS.items():
    try:
        # Current weather endpoint
        url = f"{BASE_URL}/weather"
        params = {
            'lat': stadium_info['lat'],
            'lon': stadium_info['lon'],
            'appid': API_KEY,
            'units': 'imperial'  # Fahrenheit, mph
        }
        
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        
        data = response.json()
        
        # Extract weather data
        main = data.get('main', {})
        wind = data.get('wind', {})
        weather = data.get('weather', [{}])[0]
        
        weather_record = {
            'team': team,
            'city': stadium_info['city'],
            'is_dome': stadium_info['is_dome'],
            'timestamp': datetime.fromtimestamp(data.get('dt', 0)).isoformat(),
            'temp_f': main.get('temp'),
            'feels_like_f': main.get('feels_like'),
            'temp_min_f': main.get('temp_min'),
            'temp_max_f': main.get('temp_max'),
            'pressure': main.get('pressure'),
            'humidity': main.get('humidity'),
            'wind_speed_mph': wind.get('speed'),
            'wind_deg': wind.get('deg'),
            'wind_gust_mph': wind.get('gust'),
            'visibility_meters': data.get('visibility'),
            'clouds_pct': data.get('clouds', {}).get('all'),
            'condition_main': weather.get('main'),
            'condition_desc': weather.get('description')
        }
        
        current_weather_data.append(weather_record)
        
        # Display key metrics
        temp = weather_record['temp_f']
        wind = weather_record['wind_speed_mph'] or 0
        gust = weather_record['wind_gust_mph'] or 0
        print(f"✓ {team}: {temp:.1f}°F, Wind {wind:.1f}mph, Gust {gust:.1f}mph")
        
        # Rate limit respect
        time.sleep(rate_limit_delay)
        
    except Exception as e:
        print(f"✗ {team}: Error - {e}")

if current_weather_data:
    current_weather_df = pd.DataFrame(current_weather_data)
    print(f"\n✓ Fetched current weather for {len(current_weather_df)} stadiums")
    display(current_weather_df[['team', 'city', 'temp_f', 'wind_speed_mph', 'wind_gust_mph', 'condition_desc']].head(10))
else:
    print("\n⚠️ No weather data fetched")
    current_weather_df = pd.DataFrame()

In [0]:
# Fetch 5-day forecast (3-hour intervals)
# Free tier provides forecast every 3 hours for 5 days

print("\nFetching 5-day forecast (sample of 5 stadiums)...\n")

forecast_data = []
sample_teams = ['BUF', 'GB', 'KC', 'NE', 'CHI']  # Cold weather teams

for team in sample_teams:
    if team in NFL_STADIUM_COORDS:
        stadium_info = NFL_STADIUM_COORDS[team]
        
        try:
            url = f"{BASE_URL}/forecast"
            params = {
                'lat': stadium_info['lat'],
                'lon': stadium_info['lon'],
                'appid': API_KEY,
                'units': 'imperial',
                'cnt': 40  # 5 days * 8 (3-hour intervals)
            }
            
            response = requests.get(url, params=params, timeout=10)
            response.raise_for_status()
            
            data = response.json()
            forecast_list = data.get('list', [])
            
            for forecast_item in forecast_list:
                main = forecast_item.get('main', {})
                wind = forecast_item.get('wind', {})
                weather = forecast_item.get('weather', [{}])[0]
                
                forecast_record = {
                    'team': team,
                    'city': stadium_info['city'],
                    'is_dome': stadium_info['is_dome'],
                    'forecast_dt': datetime.fromtimestamp(forecast_item.get('dt', 0)).isoformat(),
                    'temp_f': main.get('temp'),
                    'feels_like_f': main.get('feels_like'),
                    'wind_speed_mph': wind.get('speed'),
                    'wind_deg': wind.get('deg'),
                    'wind_gust_mph': wind.get('gust'),
                    'pop': forecast_item.get('pop', 0) * 100,  # Probability of precipitation %
                    'rain_3h': forecast_item.get('rain', {}).get('3h', 0),
                    'snow_3h': forecast_item.get('snow', {}).get('3h', 0),
                    'condition_main': weather.get('main'),
                    'condition_desc': weather.get('description')
                }
                
                forecast_data.append(forecast_record)
            
            print(f"✓ {team}: {len(forecast_list)} forecast intervals")
            time.sleep(rate_limit_delay)
            
        except Exception as e:
            print(f"✗ {team}: Error - {e}")

if forecast_data:
    forecast_df = pd.DataFrame(forecast_data)
    print(f"\n✓ Fetched {len(forecast_df)} forecast records")
    display(forecast_df[['team', 'forecast_dt', 'temp_f', 'wind_speed_mph', 'wind_gust_mph', 'pop']].head(15))
else:
    print("\n⚠️ No forecast data fetched")
    forecast_df = pd.DataFrame()

In [0]:
# Calculate fantasy impact scores

if 'current_weather_df' in locals() and len(current_weather_df) > 0:
    print("Calculating weather impact scores...\n")
    
    df = current_weather_df.copy()
    
    # Fill NaN values for calculations
    df['wind_speed_mph'] = df['wind_speed_mph'].fillna(0)
    df['wind_gust_mph'] = df['wind_gust_mph'].fillna(0)
    
    # Wind impact scoring
    df['wind_impact_score'] = df.apply(lambda row:
        0 if row['is_dome'] else
        1 if row['wind_speed_mph'] < 10 else
        2 if row['wind_speed_mph'] < 15 else
        3 if row['wind_speed_mph'] < 20 else
        4,
        axis=1
    )
    
    # Gust impact
    df['gust_impact_score'] = df.apply(lambda row:
        0 if row['is_dome'] else
        1 if row['wind_gust_mph'] < 15 else
        2 if row['wind_gust_mph'] < 20 else
        3 if row['wind_gust_mph'] < 25 else
        4,
        axis=1
    )
    
    # Cold impact
    df['cold_impact_score'] = df.apply(lambda row:
        0 if row['is_dome'] else
        1 if row['temp_f'] >= 40 else
        2 if row['temp_f'] >= 32 else
        3 if row['temp_f'] >= 20 else
        4,
        axis=1
    )
    
    # Overall impact
    df['overall_weather_impact'] = (
        df['wind_impact_score'] + 
        df['gust_impact_score'] + 
        df['cold_impact_score']
    )
    
    # Position adjustments (percentage)
    df['qb_adjustment'] = df['wind_impact_score'] * -0.05
    df['rb_adjustment'] = df['cold_impact_score'] * 0.03
    df['wr_adjustment'] = df['wind_impact_score'] * -0.08
    df['k_adjustment'] = (df['wind_impact_score'] + df['gust_impact_score']) * -0.10
    
    print("✓ Calculated impact scores\n")
    print("High-impact weather locations:")
    high_impact = df[df['overall_weather_impact'] >= 5].sort_values('overall_weather_impact', ascending=False)
    if len(high_impact) > 0:
        display(high_impact[['team', 'city', 'temp_f', 'wind_speed_mph', 'wind_gust_mph', 'overall_weather_impact']])
    else:
        print("No high-impact weather currently")
    
    weather_impact_df = df
else:
    print("⚠️ No weather data to score")

In [0]:
# Transform to Spark DataFrame

if 'weather_impact_df' in locals() and len(weather_impact_df) > 0:
    print("Transforming to Spark DataFrame...\n")
    
    rows = []
    for idx, row in weather_impact_df.iterrows():
        spark_row = Row(
            team=row['team'],
            city=row['city'],
            is_dome=bool(row['is_dome']),
            timestamp=row['timestamp'],
            season=SEASON,
            week=WEEK,
            temp_f=float(row['temp_f']) if pd.notna(row['temp_f']) else None,
            feels_like_f=float(row['feels_like_f']) if pd.notna(row['feels_like_f']) else None,
            wind_speed_mph=float(row['wind_speed_mph']) if pd.notna(row['wind_speed_mph']) else None,
            wind_deg=int(row['wind_deg']) if pd.notna(row['wind_deg']) else None,
            wind_gust_mph=float(row['wind_gust_mph']) if pd.notna(row['wind_gust_mph']) else None,
            humidity=int(row['humidity']) if pd.notna(row['humidity']) else None,
            visibility_meters=int(row['visibility_meters']) if pd.notna(row['visibility_meters']) else None,
            clouds_pct=int(row['clouds_pct']) if pd.notna(row['clouds_pct']) else None,
            condition_main=str(row['condition_main']) if pd.notna(row['condition_main']) else None,
            condition_desc=str(row['condition_desc']) if pd.notna(row['condition_desc']) else None,
            wind_impact_score=int(row['wind_impact_score']),
            gust_impact_score=int(row['gust_impact_score']),
            cold_impact_score=int(row['cold_impact_score']),
            overall_weather_impact=int(row['overall_weather_impact']),
            qb_adjustment=float(row['qb_adjustment']),
            rb_adjustment=float(row['rb_adjustment']),
            wr_adjustment=float(row['wr_adjustment']),
            k_adjustment=float(row['k_adjustment']),
            raw_data=json.dumps(row.to_dict(), default=str)
        )
        rows.append(spark_row)
    
    weather_spark_df = spark.createDataFrame(rows)
    print(f"✓ Created Spark DataFrame with {weather_spark_df.count()} records\n")
    display(weather_spark_df.limit(10))
else:
    print("⚠️ No data to transform")

In [0]:
# Write to bronze_nfl_weather table

if 'weather_spark_df' in locals():
    print("Writing to bronze_nfl_weather...\n")
    
    bronze_weather = weather_spark_df.withColumn("ingested_at", F.current_timestamp())
    bronze_weather = bronze_weather.withColumn("source", F.lit("openweathermap"))
    
    bronze_weather.createOrReplaceTempView("openweathermap_bronze_updates")
    
    # Merge with existing bronze table (created by weatherapi notebook)
    spark.sql("""
        MERGE INTO main.fantasai.bronze_nfl_weather AS target
        USING openweathermap_bronze_updates AS source
        ON target.team = source.team 
            AND target.season = source.season 
            AND target.week = source.week
            AND target.source = source.source
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    
    print(f"✓ Merged {bronze_weather.count()} weather records into bronze_nfl_weather\n")
else:
    print("⚠️ No data to write")

In [0]:
%sql
-- Compare OpenWeatherMap vs WeatherAPI data
SELECT 
    team,
    source,
    temp_f,
    wind_speed_mph AS wind_mph,
    wind_gust_mph AS gust_mph,
    overall_weather_impact,
    qb_adjustment,
    k_adjustment
FROM main.fantasai.bronze_nfl_weather
WHERE season = 2024 AND week = 18
ORDER BY team, source

## OpenWeatherMap Features

### Available Endpoints (Free Tier)
1. **Current Weather** - `/weather` - Real-time conditions
2. **5-Day Forecast** - `/forecast` - 3-hour intervals
3. **Historical** - Requires paid plan

### Pricing
- **Free**: 1,000 calls/day, 60 calls/min
- **Startup**: $40/month - 100K calls/day
- **Developer**: $180/month - 1M calls/day

### vs WeatherAPI.com

| Feature | OpenWeatherMap (Free) | WeatherAPI.com (Free) |
|---------|----------------------|----------------------|
| Daily Calls | 1,000 | 1,000,000 |
| Forecast | 5-day/3-hour | 3-day |
| Wind Gusts | ✓ Yes | ✓ Yes |
| Historical | ❌ Paid only | ✓ Limited free |
| Rate Limit | 60/min | No limit |
| Ease of Use | Good | Excellent |

**Recommendation:** 
- **Primary**: WeatherAPI.com (1M calls, better for production)
- **Backup**: OpenWeatherMap (good for redundancy)
- **Use both**: Cross-validate weather data

### API Key Activation
⚠️ **Important**: After signing up, API key activation takes 10 minutes to 2 hours. Test your key before deploying to production.

### Rate Limit Management
```python
import time

# Stay under 60 calls/min
for stadium in stadiums:
    fetch_weather(stadium)
    time.sleep(1.1)  # ~55 calls/min
```

### Best Practices
1. **Cache weather data** - Update every 30-60 minutes, not per request
2. **Use coordinates** - More accurate than city names
3. **Monitor your usage** - Dashboard shows call counts
4. **Handle errors gracefully** - API can be unavailable
5. **Combine with WeatherAPI.com** - Use both for redundancy